# 11.3 生成增强 (Generation Enhancement)

检索到文档后，如何有效利用检索结果增强生成质量是RAG系统的关键环节。

本节涵盖：
- Prompt工程与上下文注入
- 引用生成与幻觉检测
- 自适应检索（FLARE/Self-RAG）
- 上下文窗口管理

## 1. Prompt工程与上下文注入

**目的**：设计有效的Prompt模板，让LLM基于检索内容生成高质量回答

**Prompt模板设计原则**：
- **明确指令**：告诉模型"基于以下文档回答问题"
- **引用标记**：要求模型标注信息来源，如[1][2]
- **不确定处理**：指示模型在文档中没有答案时说"我不知道"
- **结构化输出**：要求模型按特定格式输出

**上下文注入策略**：
- **简单拼接**：documents + question → LLM
- **带编号拼接**：[1] doc1 [2] doc2 → 便于引用
- **分层注入**：摘要层+详细层，按需展开
- **重排序注入**：最相关文档放在最前面（Lost in the Middle问题）

In [ ]:
class RAGPromptBuilder:
    def __init__(self, system_prompt='', citation_style='bracket'):
        self.system_prompt = system_prompt or 'You are a helpful assistant. Answer based on the provided documents.'
        self.citation_style = citation_style

    def build(self, query, documents, max_context_tokens=2000):
        context_parts = []
        total_tokens = 0
        for i, doc in enumerate(documents):
            doc_tokens = len(doc.split())
            if total_tokens + doc_tokens > max_context_tokens:
                remaining = max_context_tokens - total_tokens
                if remaining > 50:
                    truncated = ' '.join(doc.split()[:remaining]) + '...'
                    context_parts.append(f'[{i+1}] {truncated}')
                break
            context_parts.append(f'[{i+1}] {doc}')
            total_tokens += doc_tokens

        context = '\n\n'.join(context_parts)
        prompt = f"""{self.system_prompt}

Documents:
{context}

Question: {query}

Instructions:
- Answer based ONLY on the provided documents
- Cite sources using [1], [2], etc.
- If the documents don't contain the answer, say "I cannot answer based on the provided documents."

Answer:"""
        return prompt

documents = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a subset of machine learning that uses neural networks with multiple layers to analyze various factors of data.",
    "Supervised learning requires labeled training data, while unsupervised learning discovers patterns in unlabeled data.",
    "Reinforcement learning trains agents to make decisions by rewarding desired behaviors and punishing undesired ones.",
]

builder = RAGPromptBuilder()
query = "What is the difference between supervised and unsupervised learning?"
prompt = builder.build(query, documents)

print('=== RAG Prompt Engineering ===')
print(prompt)
print(f'\nPrompt length: {len(prompt.split())} tokens')
print(f'\nKey: Good prompts explicitly instruct the model to use only provided documents.')
print(f'Citation markers [1][2] enable verification and traceability.')
print(f'Context window management prevents overflow with long documents.')

## 2. 引用生成与幻觉检测

**目的**：确保生成内容有据可依，检测和减少幻觉

**引用生成**：模型在回答中标注信息来源，如"机器学习是AI的子集[1]"。引用必须能映射回检索文档。

**幻觉类型**：
- **内在幻觉**：与检索文档矛盾（如文档说A，模型说非A）
- **外在幻觉**：检索文档未提及的信息（模型编造了文档中没有的内容）
- **错误归因**：引用了错误的文档编号

**幻觉检测方法**：
- **NLI验证**：用自然语言推理模型检查生成内容是否被文档蕴含
- **引用验证**：检查每个引用标记是否确实对应文档中的内容
- **自一致性**：多次生成取交集，不一致的部分可能是幻觉

In [ ]:
import re

class CitationVerifier:
    def __init__(self, documents):
        self.documents = documents

    def extract_claims_with_citations(self, answer):
        pattern = r'([^\[]+?)\[(\d+)\]'
        matches = re.findall(pattern, answer)
        claims = []
        for claim_text, cite_num in matches:
            cite_idx = int(cite_num) - 1
            claims.append({'claim': claim_text.strip(), 'citation': int(cite_num), 'doc_idx': cite_idx})
        return claims

    def verify_claim(self, claim, doc_idx):
        if doc_idx >= len(self.documents):
            return False, 'Citation index out of range'
        doc = self.documents[doc_idx].lower()
        claim_words = set(claim.lower().split())
        doc_words = set(doc.split())
        overlap = claim_words & doc_words
        overlap_ratio = len(overlap) / max(len(claim_words), 1)
        if overlap_ratio > 0.3:
            return True, f'Overlap: {overlap_ratio:.0%}'
        return False, f'Low overlap: {overlap_ratio:.0%}'

    def verify_answer(self, answer):
        claims = self.extract_claims_with_citations(answer)
        results = []
        for c in claims:
            valid, reason = self.verify_claim(c['claim'], c['doc_idx'])
            results.append({**c, 'valid': valid, 'reason': reason})
        return results

documents = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "Deep learning uses neural networks with multiple layers for feature extraction and transformation.",
    "Supervised learning requires labeled training data to learn input-output mappings.",
]

verifier = CitationVerifier(documents)

answer = "Machine learning is a branch of AI that enables learning from data[1]. Deep learning uses multi-layer neural networks[2]. Reinforcement learning trains agents through rewards[3]."

print('=== Citation Verification & Hallucination Detection ===')
print(f'Answer: {answer}')
print(f'\nVerification results:')
results = verifier.verify_answer(answer)
for r in results:
    status = '✓ Verified' if r['valid'] else '✗ Hallucination'
    print(f'  [{r["citation"]}] "{r["claim"][:50]}" → {status} ({r["reason"]})')

n_verified = sum(1 for r in results if r['valid'])
print(f'\nVerified: {n_verified}/{len(results)} claims')
print(f'\nKey: Citation verification catches hallucinations by checking claims against source documents.')
print(f'Production systems use NLI models for more accurate verification.')
print(f'Always instruct the model to cite sources and say "I don\'t know" when uncertain.')

## 3. 自适应检索（FLARE/Self-RAG）

**目的**：让模型自主决定何时检索、检索什么，提高效率和准确性

**FLARE（Forward-Looking Active REtrieval）**：
- 模型先生成初步回答
- 检测低置信度token（概率低于阈值）
- 用低置信度部分作为新查询检索
- 用检索结果重新生成该部分

**Self-RAG**：
- 检索反思token：[Retrieve]/[No Retrieve] → 是否需要检索
- 相关性反思token：[Relevant]/[Irrelevant] → 检索结果是否相关
- 支持度反思token：[Supported]/[Not Supported] → 生成是否有据

**工业实践**：
- 简单事实查询：不需要检索（模型已知）
- 时效性查询：必须检索（模型训练截止后的事实）
- 复杂推理：多轮检索+推理交替

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class FLAREDecider(nn.Module):
    def __init__(self, d_model=64, threshold=0.3):
        super().__init__()
        self.threshold = threshold
        self.confidence_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, hidden_states):
        confidence = self.confidence_head(hidden_states).squeeze(-1)
        need_retrieval = confidence < self.threshold
        return confidence, need_retrieval

class SelfRAGTokenClassifier(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()
        self.retrieve_cls = nn.Linear(d_model, 2)
        self.relevant_cls = nn.Linear(d_model, 2)
        self.support_cls = nn.Linear(d_model, 2)

    def forward(self, hidden_state):
        retrieve = self.retrieve_cls(hidden_state).argmax(-1)
        relevant = self.relevant_cls(hidden_state).argmax(-1)
        support = self.support_cls(hidden_state).argmax(-1)
        return retrieve, relevant, support

print('=== Adaptive Retrieval: FLARE vs Self-RAG ===')

flare = FLAREDecider(threshold=0.5)
states = torch.randn(10, 64)
confidences, decisions = flare(states)

print(f'\n--- FLARE: Token-level Retrieval Decision ---')
print(f'Token confidences: {[f"{c:.2f}" for c in confidences.tolist()]}')
low_conf_tokens = [i for i, d in enumerate(decisions.tolist()) if d]
print(f'Low-confidence tokens (need retrieval): {low_conf_tokens}')
print(f'Retrieval rate: {len(low_conf_tokens)}/{len(states)} ({len(low_conf_tokens)/len(states):.0%})')

self_rag = SelfRAGTokenClassifier()
state = torch.randn(1, 64)
retrieve, relevant, support = self_rag(state)

print(f'\n--- Self-RAG: Reflection Tokens ---')
retrieve_label = ['No Retrieve', 'Retrieve'][retrieve.item()]
relevant_label = ['Irrelevant', 'Relevant'][relevant.item()]
support_label = ['Not Supported', 'Supported'][support.item()]
print(f'  Retrieve decision: {retrieve_label}')
print(f'  Relevance check:   {relevant_label}')
print(f'  Support check:     {support_label}')

print(f'\n--- Adaptive Retrieval Decision Tree ---')
print(f'1. Is the query about recent events? → Always retrieve')
print(f'2. Is the model confident in its answer? → Skip retrieval (FLARE)')
print(f'3. Does the retrieved doc seem relevant? → Use it (Self-RAG [Relevant])')
print(f'4. Is the answer supported by the doc? → Keep it (Self-RAG [Supported])')

print(f'\nKey: FLARE triggers retrieval only when model confidence is low.')
print(f'Self-RAG adds reflection tokens for retrieval, relevance, and support checks.')
print(f'Both reduce unnecessary retrieval and improve answer quality.')